# Final Milestone - LLM Comparison and Tool Demo

For the final milestone, we have decided to do the following:
1. Load retrievers and RAG pipeline components
2. Compare two LLMs: `Meta-Llama-3-8B-Instruct` (HF API) vs `Qwen3 1.7B` (local via Ollama)
3. Run 5 queries through both models with identical retriever and prompt
4. Demonstrate Tavily web search tool on 3 example queries

**Note:** We initially planned to run both models via the HF Inference API, but `Qwen/Qwen3.5-2B` is not hosted by any HF provider. We use Ollama to run Qwen3 locally instead, which also gives us an interesting API-vs-local comparison angle.

In [12]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

True

In [13]:
from src.bm25 import BM25Retriever
from src.semantic import SemanticRetriever

INDICES = Path.cwd().parent / "indices"

bm25 = BM25Retriever()
bm25.load(str(INDICES / "bm25_index.pkl"))

semantic = SemanticRetriever()
semantic.load(str(INDICES / "faiss_index"))

print(f"BM25 corpus: {len(bm25.corpus):,} products")
print(f"Semantic corpus: {len(semantic.corpus):,} products")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3154.63it/s]


BM25 corpus: 112,590 products
Semantic corpus: 112,590 products


## Step 1.2: LLM Comparison

We compare two models using **identical** retrieval (Hybrid/RRF) and prompt (`strict_citation`):

| Model | Parameters | Family | Provider |
|---|---|---|---|
| `meta-llama/Meta-Llama-3-8B-Instruct` | 8B | Meta Llama 3 | HF Inference API (remote) |
| `qwen3:1.7b` | 1.7B | Alibaba Qwen 3 | Ollama (local) |

The ~5x parameter difference and different model families should reveal quality-vs-size tradeoffs in citation accuracy, completeness, and fluency.

In [14]:
from src.rag_pipeline import RAGPipeline, load_llm
from langchain_ollama import ChatOllama

MODEL_A = "meta-llama/Meta-Llama-3-8B-Instruct"
MODEL_B = "qwen3:1.7b"

llm_a = load_llm(model_id=MODEL_A, temperature=0, seed=575)
llm_b = ChatOllama(model=MODEL_B, temperature=0, seed=575)

print(f"Model A: {MODEL_A} (HF Inference API)")
print(f"Model B: {MODEL_B} (Ollama, local)")

Model A: meta-llama/Meta-Llama-3-8B-Instruct (HF Inference API)
Model B: qwen3:1.7b (Ollama, local)


In [15]:
pipeline_a = RAGPipeline(
    bm25=bm25, semantic=semantic,
    retriever_name="Hybrid", prompt_name="strict_citation",
    llm=llm_a, top_k=5,
)
pipeline_b = RAGPipeline(
    bm25=bm25, semantic=semantic,
    retriever_name="Hybrid", prompt_name="strict_citation",
    llm=llm_b, top_k=5,
)

comparison_queries = [
    "vitamin C serum for brightening",
    "what helps with mild acne and post-acne marks?",
    "gentle face wash that won't irritate sensitive skin",
    "best skincare routine for oily acne-prone teenage skin under $30",
    "what helps with sun damage on fair skin around the eyes",
]

results_comparison = []


def run_comparison(query: str) -> None:
    """Run one query through both pipelines, print both answers, record the result."""
    result_a = pipeline_a.answer(query)
    result_b = pipeline_b.answer(query)

    print(f"QUERY: {query}\n")
    print("--- Model A (Llama-3-8B, HF API) ---")
    print(result_a["answer"])
    print()
    print("--- Model B (Qwen3 1.7B, Ollama) ---")
    print(result_b["answer"])

    results_comparison.append(
        {
            "query": query,
            "answer_a": result_a["answer"],
            "answer_b": result_b["answer"],
            "sources_a": result_a["sources"],
            "sources_b": result_b["sources"],
        }
    )


### Query 1 - `"vitamin C serum for brightening"`

Direct keyword query against a well-represented product category. Both models should have strong retrieval hits, so differences here are about answer style rather than grounding.

In [16]:
run_comparison(comparison_queries[0])

QUERY: vitamin C serum for brightening

--- Model A (Llama-3-8B, HF API) ---
Based on the provided reviews and metadata, here are some vitamin C serums that are suitable for brightening:

1. [B01M1OZ560] - Charlotte Elizabeth Organics Advanced Illuminating Vitamin C Serum: The reviewer mentioned that they and their wife both like this product, implying that it has a brightening effect.
2. [B0874HTLRB] - Essano Vitamin C Brightening Serum - Boost and Brighten, 30ml: The reviewer mentioned that the product is refreshing and smells amazing, suggesting that it may be effective for brightening.
3. [B094C2M597] - Trilogy Vitamin C Brightening Set - For Dull Skin - Made in New Zealand: The reviewer liked that the product was unscented and absorbed quickly into their skin, suggesting that it may be effective for brightening.
4. [B01N6ADBNM] - Mererke_Pretty Vitamin C Serum for Face: The reviewer mentioned that their wife loves this product, implying that it has a brightening effect.
5. [B09]  

**Discussion - Query 1**

Config: `temperature=0, seed=575` for both models (`do_sample=True` default on the HF endpoint).

Both models return the same three products as their top recommendations - **Charlotte Elizabeth** (`B01M1OZ560`), **Essano** (`B0874HTLRB`), **Trilogy** (`B094C2M597`). After that they diverge:

- **Llama-3-8B (HF API)** returns a numbered prose list of 7 entries with `Mererke` (`B01N6ADBNM`), `Makana` (`B098FK3SNH`), a Vitamin C + Hyaluronic Acid serum (`B07TVP8XK8`), and a repeat of Essano. It cites ASINs plus a short review quote per product, but does not return any `Rating: X.X/5` values even though the prompt provides them in context. It also produces a small formatting artifact (`"[B09] is missing for..."`) before completing the Makana entry.
- **Qwen3 1.7B (Ollama)** returns a bold-header bullet catalog of 4 entries (the top 3 plus Eva St. Claire `B07PC2CNK7`), each with the metadata rating rendered as a `Rating: X.X/5` field. It closes with a one-line synthesis naming Charlotte Elizabeth as the strongest pick.

So on this query the split is **format and metadata usage** rather than grounding. Llama is quote-driven and wider in coverage; Qwen is rating-driven and more decision-ready.

### Query 2 - `"what helps with mild acne and post-acne marks?"`

Natural-language question covering two related concerns (active acne and post-acne marks). Tests how each model synthesizes advice when the user asks about more than one issue.

In [17]:
run_comparison(comparison_queries[1])

QUERY: what helps with mild acne and post-acne marks?

--- Model A (Llama-3-8B, HF API) ---
Based on the provided reviews, the following products may help with mild acne and post-acne marks:

1. [B07FPG7MXW] Tea-Tree Oil in the Acne Patch 3 Sheets (36 Counts per Sheet) Hydrocolloid Pimple Blemish Cover Treatment with CICA Tea-Tree Oil may help with acne, and 
2. [B07FPG7MXW] The patches also helped to remove redness and the inside of the pimple, which can be considered as post-acne marks.
3. [B01CKW44XY] Tepezcohuite Cream Day and Night Set de 2 additioned with "Colagen & Vitamin E" has helped a customer with severe acne for over 35 years to finally have a clear face.
4. [B0759S2PZH] Organic Tamanu Seed Oil & Roller, Cold Pressed & Unrefined has helped a customer's skin improve with mild acne.
5. [B00M0YRNWC] Max Factor Colour Corrector Sticks - GREEN has helped a customer to fade dark spots from post-acne.
6. [B01IA9D07M] Burts Acne Spt Trtmnt Size .26z has helped a customer to reduce

**Discussion - Query 2**

Both models stay grounded on ASINs and there is a three-product overlap in their recommendations: **Tepezcohuite Cream** (`B01CKW44XY`), **Organic Tamanu Seed Oil** (`B0759S2PZH`), and **Burts Acne Spot Treatment** (`B01IA9D07M`). The divergence is what *else* each model adds to the list.

- **Llama-3-8B** adds the tea-tree / hydrocolloid acne patches (`B07FPG7MXW`, cited twice) and the Max Factor green color corrector (`B00M0YRNWC`) as a dark-spot fader. It cites review snippets only with no ratings returned. Closes with the hedge *"These products may not work for everyone, and individual results may vary."*
- **Qwen3 1.7B** adds the strawberry & papaya sugar scrub (`B09BJK8WKP`, 4.7/5), Finale Clear Patch (`B009KVOVLE`, 4.0/5), and Natural Acne Scar Removal Cream (`B01I05IRII`, 2.9/5). Every entry carries a `Rating: X.X/5`. Qwen does surface the low 2.9/5 rating, but then softens it in prose (*"While not the highest-rated, it is noted as a good product for scar and stretch mark removal"*) — arguably papering over the rating signal it just cited.

Same context, same retrieval, but Llama defaults to qualitative review evidence and Qwen defaults to numeric ratings. Qwen's quantitative presentation is easier to scan and Llama's tone carries more nuance.

### Query 3 - `"gentle face wash that won't irritate sensitive skin"`

Constraint-driven query. The LLM must filter cited products by gentleness / sensitivity signals in the review text, not just keyword match the product title.

In [18]:
run_comparison(comparison_queries[2])

QUERY: gentle face wash that won't irritate sensitive skin

--- Model A (Llama-3-8B, HF API) ---
Based on the reviews, the following products are recommended for sensitive skin:

1. ASIN: B0741FX68F - Natural Face Wash for Sensitive Skin by Eve Hansen: Gentle on sensitive skin, non-drying, and has anti-aging properties with Vitamin C and Vitamin E.
2. ASIN: B01JA7TMSO - Calming Chamomile Daily Face Cleanser: Gentle, natural ingredients, and doesn't irritate sensitive skin.
3. ASIN: B075WYW1GL - Medimix Ayurvedic Face Wash: Clears up mild acne, doesn't dry out sensitive skin, and has a calming effect.
4. ASIN: B012H56FTK - Medicalia Gentle Cleanser: Good for sensitive skin, effective, and gentle.
5. ASIN: B07P3GZ57Z - THE CREME SHOP BEAUTY WATER (I am BALANCED): Great product for sensitive skin, gentle, and non-irritating.
6. ASIN: B06XS2PLWX - Green Tea Foam Cleansing by 3W Clinic: Gentle on skin, non-reactive, and suitable for sensitive skin.
7. ASIN: B082J44X3J - Joli Noir Glycolic A

**Discussion - Query 3**

Strong overlap on this query. Both models return the same cluster of sensitive-skin cleansers - THE CREME SHOP Beauty Water (`B07P3GZ57Z`), 3W Clinic Green Tea Foam (`B06XS2PLWX`), Medicalia (`B012H56FTK`), Medimix Ayurvedic (`B075WYW1GL`), Calming Chamomile (`B01JA7TMSO`), Joli Noir (`B082J44X3J`), and Moroccan Rose Water Cleanser (`B07T4JF9M7`).

- **Llama-3-8B** lists 9 products (adding Eve Hansen `B0741FX68F` and Olay Gentle Clean `B0167JZ32M`). It again **drops the rating field entirely** and relies on short paraphrased descriptions (*"Gentle on sensitive skin, non-drying"*, *"Clears up mild acne, doesn't dry out sensitive skin"*).
- **Qwen3 1.7B** lists 7 products with a `Rating: X.X/5` on every entry.

Across Q1-Q3 this is now a consistent pattern: **Qwen uses ratings on every answer, Llama uses them on none.** Given both models saw the same context with the same rating fields attached, this looks like a stable stylistic / instruction-following difference between the two models at this scale, not retrieval noise.

### Query 4 - `"best skincare routine for oily acne-prone teenage skin under $30"`

Multi-constraint query combining skin type, age group, and a price ceiling. The price constraint is a good hallucination test - the retrieved context does not always include prices.

In [19]:
run_comparison(comparison_queries[3])

QUERY: best skincare routine for oily acne-prone teenage skin under $30

--- Model A (Llama-3-8B, HF API) ---
Based on the provided reviews and metadata, I would recommend a skincare routine consisting of a gentle cleanser and a salicylic acid-based product. 

For a gentle cleanser, consider the Rohto Acnes Medical Soft Cleansing Foam 160 mL ([B007409E5Q]). It is a non-drying, hydrating cleanser suitable for sensitive, acne-prone, and combination skin.

For a product containing salicylic acid, look at the Flawless Skin; Anti-Acne/Anti-Aging Skin Clearing Gel with Salicylic Acid, Lactic Acid, Glycolic Acid, Zinc Oxide and Tea Tree Oil ([B018STZ1WG]). 

The Tropical Fruits Gentle Exfoliating Strawberry & Papaya Sugar Scrub ([B09BJK8WKP]) seems expensive, but it's worth a try. Exfoliating once or twice a week can help with acne-prone skin.

The total cost of these products is under $30.

--- Model B (Qwen3 1.7B, Ollama) ---
Based on the reviews, the best skincare routine for oily, acne-pr

**Discussion - Query 4**

The price constraint is the hallucination test on this query, and **both models fail it in different ways**.

- **Llama-3-8B** recommends a three-step routine: Rohto Acnes Cleansing Foam (`B007409E5Q`), a salicylic-acid anti-acne gel (`B018STZ1WG`), and the Tropical Fruits sugar scrub (`B09BJK8WKP`). It closes with the assertion *"The total cost of these products is under $30."* The retrieved context does not include explicit prices for these products, so this claim is ungrounded. Llama asserts the budget is met without evidence from context. It also hedges on the scrub (*"seems expensive, but it's worth a try"*) without naming a price, which makes the final "under $30" sentence internally inconsistent with its own hedge.
- **Qwen3 1.7B** builds a fuller routine but makes a **category error**: in the routine suggestions block it says *"opt for a gentle, non-oily moisturizer (e.g., `[B019PM5PRQ]` The Best Acne Skin Clearing Face Wash)"*. `B019PM5PRQ` is a face wash, not a moisturizer. Qwen reuses a cleanser ASIN to fill the moisturizer slot. It closes with *"All products are under $30 and cater to oily, acne-prone skin needs"*, the same unsupported budget claim. On the positive side, Qwen does cite the very low 1.0/5 rating on `B01L8218Q8` as a reason to avoid that product.

So on this query both models hallucinate price compliance, and Qwen adds an extra category-confusion error on top.

### Query 5 - `"what helps with sun damage on fair skin around the eyes"`

Specific concern (sun damage) + body location (eye area) + skin tone. Tests fine-grained matching against what actually appears in the review text.

In [20]:
run_comparison(comparison_queries[4])

QUERY: what helps with sun damage on fair skin around the eyes

--- Model A (Llama-3-8B, HF API) ---
Based on the reviews, the following products help with sun damage on fair skin around the eyes:

1. [B076J5S1T8] Under Eye Patches - 24k Gold Eye Mask Anti-Aging Eye Treatment - This product is described as a moisturizing treatment that leaves skin soft and smooth.
2. [B01I20S3TO] Coral Safe Natural Aloe Vera Gel - This product is a natural aloe vera gel that is biodegradable and reef friendly, but it's not specifically mentioned to help with sun damage around the eyes, however, reviews often state how this product can work as an after-sun treatment, which can help with reducing the appearance of sun damage.
3. [B007YPNKOM] Josie Maran Argan Self Tanning Cream - This product is a self-tanning cream that leaves a natural glow, but it's not a sun damage treatment or specifically used around the eyes.
4. [B09] - Although there is no specific review for a product ASIN: B09,  [B01HMX4YAO] Ve

**Discussion - Query 5**

Thin corpus coverage on this specific combination (sun damage + eye area + fair skin), and the models behave very differently.

**Model A (Llama-3-8B)** returns 4 candidates and, importantly, **hedges** on most of them when the match is weak:

- Under Eye Patches (`B076J5S1T8`) - described as a moisturizing eye treatment.
- Coral Safe Aloe Vera (`B01I20S3TO`) - explicitly flagged: *"not specifically mentioned to help with sun damage around the eyes, however, reviews often state how this product can work as an after-sun treatment."*
- Josie Maran Argan Self-Tanning Cream (`B007YPNKOM`) - also flagged: *"not a sun damage treatment or specifically used around the eyes."*
- Vertra Clear SPF 28 Face Stick (`B01HMX4YAO`) - named as the most recommended product for actually preventing sun damage.

Llama also produces a minor formatting artifact (*"[B09] - Although there is no specific review for a product ASIN: B09"*) before arriving at the Vertra recommendation. ASIN usage is consistent with the other queries: correct IDs, no ratings surfaced.

**Model B (Qwen3 1.7B)** returns 2 candidates and both have **misattributed ASINs**. Qwen writes:

- `[9] ASIN: B01CNFRQPQ (Vertra Clear SPF 28 Face Stick)` - but the correct ASIN for Vertra is `B01HMX4YAO`, as Llama's answer confirms from the same context.
- `[5] ASIN: B07WG9XCH5 (Under Eye Patches ... Anti-Aging Eye Treatment)` - but the correct ASIN for the Under Eye Patches is `B076J5S1T8`, again cross-referenced against Llama on the same context.

Both bracket numbers are present in the context listing and the product **names** are correct, but each product is paired with an ASIN from a different row of the context. Temperature/seed can't explain this — it is a citation integrity failure, and it is the single most damaging error class for a RAG system whose job is to ground claims to specific products.

---

**Overall.** Across the five queries:

- **Llama-3-8B** is more verbose and more willing to hedge or flag weak matches (the Q2 tail caveat, the explicit "not specifically for" notes in Q5). Its main weakness on this corpus is ungrounded quantitative claims - the "under $30" assertion in Q4 is pulled out of thin air, and it never uses the rating field the prompt provides (Q1–Q3 all rating-free).
- **Qwen3 1.7B** is cleaner and uses the metadata rating field uniformly across every query, which makes its answers easier to scan. But it has **reliable, repeated ASIN-attribution bugs**: category confusion in Q4 (face wash → moisturizer) and off-by-one ASIN citations in Q5. For a recommender whose whole value proposition is grounding recommendations to specific products, Qwen's bug class is the more damaging one.

**On expanding the eval.** The five queries already surface distinct, reproducible failure modes in both models. Adding more queries would mostly produce variations on the same themes rather than new information.

In [21]:
import json

COMPARISON_OUT = Path.cwd().parent / "data" / "eval_outputs" / "llm_comparison.json"
COMPARISON_OUT.parent.mkdir(parents=True, exist_ok=True)
COMPARISON_OUT.write_text(json.dumps(results_comparison, indent=2))
print(f"Saved {len(results_comparison)} comparison results to {COMPARISON_OUT}")

Saved 5 comparison results to c:\Users\amato\MDS\b6\575\DSCI_575_project_willchh_jiromig\data\eval_outputs\llm_comparison.json


## Step 2: Tool Integration Demo (Tavily Web Search)

We demonstrate the Tavily web search tool (`src/tools.py`) on 3 queries where the corpus context alone may be insufficient — e.g., queries about current prices, recent product launches, or ingredient safety information that reviews don't cover.

In [22]:
from src.tools import web_search

tool_queries = [
    "Is retinol safe to use with vitamin C serum?",
    "best drugstore sunscreen 2025 dermatologist recommended",
    "Johnson & Johnson sunscreen recall or safety issues",
]

tool_results = []
for q in tool_queries:
    print("QUERY:", q)
    print("-" * 60)

    # RAG answer (from corpus only)
    rag_result = pipeline_a.answer(q)
    print("RAG-only answer (first 300 chars):")
    print(rag_result["answer"][:300])
    print()

    # Web search result
    web_result = web_search.invoke({"query": q})
    print("Tavily web search result (first 500 chars):")
    print(web_result[:500] if web_result else "(no results)")
    print()

    tool_results.append({
        "query": q,
        "rag_answer": rag_result["answer"],
        "web_result": web_result,
    })

TOOL_OUT = Path.cwd().parent / "data" / "eval_outputs" / "tool_demo.json"
TOOL_OUT.write_text(json.dumps(tool_results, indent=2))
print(f"Saved {len(tool_results)} tool demo results to {TOOL_OUT}")

QUERY: Is retinol safe to use with vitamin C serum?
------------------------------------------------------------
RAG-only answer (first 300 chars):
I don't have enough information.

Tavily web search result (first 500 chars):
Vitamin C and retinol are two of the most high-profile ingredients in the world of skincare, thanks to the powerful skin benefits both ingredients provide. Anti-ageing properties are some of the most sought-after effects from skincare users and both retinol and vitamin C are known to help maintain the appearance of youthful skin. Given that both these ingredients can help lead to brighter, more even and younger-looking skin, it’s no surprise that questions arise about using retinol and vitamin C

QUERY: best drugstore sunscreen 2025 dermatologist recommended
------------------------------------------------------------
RAG-only answer (first 300 chars):
Based on the provided reviews, I don't have enough information to recommend the best drugstore sunscreen specific

**Discussion - Tool Integration (Tavily Web Search)**

The three tool queries were picked precisely because our corpus **cannot** answer them - they probe knowledge types that product-review data structurally doesn't contain. The contrast between the RAG-only column and the web-search column shows the tool earning its place:

**Query 1 - "Is retinol safe to use with vitamin C serum?"**
A skincare-combination / ingredient-interaction question. Individual product reviews rarely discuss how two ingredients interact, and even when one user mentions it, it's anecdotal. The RAG answer correctly hedges (*"I don't have enough information to conclude whether retinol is safe to use with vitamin C serum"*) and only offers a hedged user datapoint from review [7]. Tavily returns the actual practitioner guidance (*"Yes, you can use vitamin C with retinol... use vitamin C in the morning"*). This is knowledge the corpus fundamentally lacks, not a retrieval miss.

**Query 2 - "best drugstore sunscreen 2025 dermatologist recommended"**
Recency-bound. The corpus is a static 2023 snapshot; no amount of good retrieval can surface 2025 recommendations. RAG pivots to "dermatologist-recommended sunscreens from the provided reviews" (Coppertone etc.) which is a reasonable degraded answer but it ignores the year constraint. Tavily returns current picks from Dr. Dray and Dr. Sugai (CeraVe Hydrating Mineral SPF 30, Neutrogena Hydro Boost, EltaMD UV Skin Recovery SPF 40).

**Query 3 - "Johnson & Johnson sunscreen recall or safety issues"**
Safety/regulatory news, not review content. RAG abstains cleanly (*"I don't have enough information."*). Tavily returns the actual recall information: five Neutrogena/Aveeno aerosol sunscreens recalled due to benzene contamination, traced back to Valisure's independent lab findings. This is the cleanest demo — the RAG answer is honest that the corpus has no signal, and the web tool fills in the gap with a concrete, actionable answer.

**What this tells us about the integration pattern**

- The `strict_citation` prompt's explicit *"I don't have enough information"* escape hatch is what makes tool routing feasible - it produces a clean signal that the agent (or the UI) can use to trigger a web-search fallback instead of forcing a confabulated answer.
- Product-review corpora have systematic blind spots: ingredient interactions, recency-bound questions, and safety/regulatory news. A recommender that covers only the first case (product picks) will feel frustrating to users who ask the others. Web search is the cheapest way to cover them.
- The tool is additive, not a replacement. Queries like our five earlier comparison queries ("vitamin C serum for brightening", etc.) are still better answered by RAG - the corpus has strong, opinionated, grounded signal. A production system would **route** queries: product recommendations -> RAG, ingredient/safety/recency -> web.